# 08 — Native Apps
### The same pipeline, everywhere Python runs

---

**Format:** Narrated demo — pre-run outputs. Both options run in under 7 minutes.

This module is a standalone addendum. It answers one question: **once you have a validated, production-grade pipeline from Modules 01–07, where else can it run?**

```
Option A — PyScript    Python in the browser via WebAssembly (Pyodide)
                        Single HTML file. No server. Open in any browser.
                        numpy, matplotlib, scikit-learn all work.
                        Select a target → Run Analysis → three charts output.

Option B — BeeWare     Python as a native OS app (Briefcase + Toga)
                        briefcase dev → native macOS/Windows/Linux window.
                        Same targets, same pipeline, native widgets.
                        briefcase build → .dmg / .msi / AppImage / .ipa / .aab
                        Funded by Anaconda.
```

**The data:** Five real exoplanet systems with light curves synthesised from published TESS parameters. Same `PHASE / LC_DETREND / MODEL_INIT` schema as Module 01.

**The pipeline:** Same IsolationForest anomaly detection and validation stats as `ingestion.py` from Module 01.

**What changes:** only the delivery layer. The pipeline is portable because its outputs are typed and structured.

---
## The five targets

Both options use the same target catalogue. The data is synthesised from published orbital parameters using a limb-darkened transit model + realistic TESS photometric noise — no CSV file needed.

| Target | Type | Period | Transit depth | What makes it interesting |
|---|---|---|---|---|
| **WASP-18 b** | Hot Jupiter | 0.94 days | 1.01% | Shortest period, clearest signal — same system as Module 01 |
| **WASP-12 b** | Hot Jupiter | 1.09 days | 1.43% | Tidally disrupting, deeper transit, higher noise |
| **HD 209458 b** | Hot Jupiter | 3.52 days | 1.47% | First confirmed transiting exoplanet (1999), lowest noise |
| **TRAPPIST-1 e** | Rocky — habitable zone | 6.10 days | 0.35% | Deliberately hard — shallow signal, M dwarf host noise |
| **Kepler-7 b** | Inflated Hot Jupiter | 4.89 days | 0.83% | Styrofoam density, long transit duration, interesting residuals |

---
# Option A — PyScript

PyScript was launched by Anaconda at PyCon US 2022. It loads a full CPython interpreter (Pyodide) compiled to WebAssembly into a browser tab. No server, no backend, no install.

The entire app is `option-a-pyscript/index.html` — 688 lines, self-contained.

## Running it

In [ ]:
print("""cd option-a-pyscript
python -m http.server 8080
# Open http://localhost:8080

That's it. No CSV to copy. No packages to install.

What loads in the browser (~10 MB, cached after first visit):
  Pyodide runtime  — full CPython compiled to WebAssembly
  numpy            — bundled in Pyodide, no micropip
  pandas           — bundled in Pyodide
  matplotlib       — bundled in Pyodide
  scikit-learn     — bundled in Pyodide

NOT available in WASM (no compiled build exists):
  numba, dask, pytorch

From Anaconda Notebooks docs:
  'browser-side operations require copying all data down to the browser,
   which is impractical for large datasets. Some libraries use operations
   not available in WASM (numba, dask, pytorch).'""")

cd option-a-pyscript
python -m http.server 8080
# Open http://localhost:8080

That's it. No CSV to copy. No packages to install.

What loads in the browser (~10 MB, cached after first visit):
  Pyodide runtime  — full CPython compiled to WebAssembly
  numpy            — bundled in Pyodide, no micropip
  pandas           — bundled in Pyodide
  matplotlib       — bundled in Pyodide
  scikit-learn     — bundled in Pyodide

NOT available in WASM (no compiled build exists):
  numba, dask, pytorch

From Anaconda Notebooks docs:
  'browser-side operations require copying all data down to the browser,
   which is impractical for large datasets. Some libraries use operations
   not available in WASM (numba, dask, pytorch).'


## What the app does

1. **Step 1 — Choose a target** from the dropdown. The info card updates with the system's physical properties.
2. **Step 2 — Press Run Analysis.** The pipeline executes in the browser tab:
   - Synthesise light curve from published parameters (Mandel-Agol limb-darkened transit + TESS noise)
   - Validate schema, nulls, flux std, duplicate phases
   - IsolationForest anomaly detection (`contamination=0.05`, same params as Module 01)
   - Each step logs to the pipeline log in real time
3. **Step 3 — Validation report** populates with the same fields as Module 01's `ValidationReport`
4. **Step 4 — Three charts render:**
   - Full-width: phase-folded light curve + model + anomaly markers + residual panel
   - Half-width: IsolationForest score histogram (normal vs anomalous populations)
   - Half-width: residual coloured by anomaly score (shows transit signal concentration)

## The PyScript pattern

Everything lives in a single `<script type="py">` block. Packages are declared in the `config` attribute — Pyodide fetches and installs them before executing your code.

```html
<!-- 1. Load PyScript runtime from CDN -->
<link rel="stylesheet" href="https://pyscript.net/releases/2025.11.1/core.css" />
<script type="module" src="https://pyscript.net/releases/2025.11.1/core.js"></script>

<!-- 2. Python block — runs in Pyodide (WebAssembly CPython) -->
<script type="py" config='{"packages": ["matplotlib", "scikit-learn"]}'>
import numpy as np          # pre-bundled — no micropip needed
import pandas as pd          # pre-bundled
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from pyscript import document   # DOM access from Python

def run(event=None):
    key = document.querySelector("#target-select").value
    df  = synthesise(key)           # generate light curve
    report = validate(df)           # ValidationReport-equivalent dict
    result = detect_anomalies(df)   # IsolationForest
    inject_img("chart-lc", chart_lightcurve(df, result))  # matplotlib → base64 PNG
    document.querySelector("#stats-grid").innerHTML = render_stats(report, result)

document.querySelector("#run-btn").addEventListener("click", run)
</script>
```

Key points:
- `numpy` and `pandas` are pre-bundled — they just `import` without configuration
- `matplotlib` and `scikit-learn` have WASM-compiled C extensions — declare them in `config`
- `document.querySelector()` accesses the HTML DOM directly from Python
- matplotlib figures render as base64-encoded PNGs injected into `<img>` tags
- Everything after initial load runs offline — no network calls

---
# Option B — BeeWare

BeeWare is a suite of Python tools for building native apps. Briefcase handles packaging. Toga provides a widget toolkit where each Python widget maps to native OS controls:

| Toga widget | macOS | Windows | Linux |
|---|---|---|---|
| `toga.Label` | `NSTextField` | `Static` | `GtkLabel` |
| `toga.Button` | `NSButton` | `Button` | `GtkButton` |
| `toga.Selection` | `NSPopUpButton` | `ComboBox` | `GtkComboBoxText` |
| `toga.Table` | `NSTableView` | `ListView` | `GtkTreeView` |
| `toga.MultilineTextInput` | `NSTextView` | `Edit` | `GtkTextView` |

Briefcase packages the same Python source into native apps for every platform. Anaconda funds the project and is actively building iOS/Android binary wheels for numpy and scikit-learn.

## Running it

In [ ]:
print("""cd option-b-beeware
pip install briefcase

# Development mode — fastest, no build step, opens immediately:
briefcase dev

# Build and run a packaged app for the current OS:
briefcase create
briefcase build
briefcase run

# Package a distributable installer:
briefcase package --adhoc-sign
# → macOS:   Lightcurve Explorer-0.1.0.dmg
# → Windows: Lightcurve Explorer-0.1.0.msi
# → Linux:   Lightcurve_Explorer-0.1.0-x86_64.AppImage""")

cd option-b-beeware
pip install briefcase

# Development mode — fastest, no build step, opens immediately:
briefcase dev

# Build and run a packaged app for the current OS:
briefcase create
briefcase build
briefcase run

# Package a distributable installer:
briefcase package --adhoc-sign
# → macOS:   Lightcurve Explorer-0.1.0.dmg
# → Windows: Lightcurve Explorer-0.1.0.msi
# → Linux:   Lightcurve_Explorer-0.1.0-x86_64.AppImage


## What the app does

The UI mirrors Option A exactly — same targets, same pipeline steps, same results:

1. **`toga.Selection`** dropdown — choose a stellar target. The description label updates.
2. **`toga.Button` → Run Analysis** — runs the four pipeline steps with live log output:
   - `[1/4]` Synthesise light curve
   - `[2/4]` Validate schema + stats
   - `[3/4]` IsolationForest anomaly detection (falls back to 2.5σ threshold if numpy unavailable — mobile compatibility)
   - `[4/4]` Populate native widgets
3. **`toga.MultilineTextInput`** (readonly) — pipeline log, scrollable
4. **Validation report** — `toga.Label` pairs in a `Box(COLUMN)`
5. **`toga.Table`** — anomalous points, sorted by score, with native column headers

## The Toga pattern

```python
import toga
from toga.style import Pack
from toga.style.pack import COLUMN, ROW

class LightcurveApp(toga.App):
    def startup(self):
        # All widgets are native OS controls on every platform
        selector = toga.Selection(
            items=["WASP-18 b", "WASP-12 b", "HD 209458 b", "TRAPPIST-1 e", "Kepler-7 b"],
            on_change=self._on_target_change,
        )
        run_btn = toga.Button("▶  Run Analysis", on_press=self._on_run)
        log     = toga.MultilineTextInput(readonly=True, style=Pack(height=140))
        table   = toga.Table(headings=["PHASE", "LC_DETREND", "MODEL_INIT", "Score"])

        # Box layout — Pack(direction=COLUMN) or ROW
        outer = toga.Box(
            children=[selector, run_btn, log, table],
            style=Pack(direction=COLUMN, padding=16, gap=12),
        )
        self.main_window = toga.MainWindow(title="Lightcurve Explorer", size=(860, 680))
        self.main_window.content = toga.ScrollContainer(content=outer, horizontal=False)
        self.main_window.show()

    def _on_run(self, widget):
        key = TARGET_KEYS[TARGET_NAMES.index(self._selector.value)]
        phase, lc, model = synthesise(key)       # same pipeline as ingestion.py
        report = validate(phase, lc, model)      # same fields as ValidationReport
        result = detect_anomalies(phase, lc, model)  # IsolationForest
        self._table.data = [
            (f'{r[0]:+.5f}', f'{r[1]:.7f}', f'{r[2]:.7f}', f'{r[3]:.4f}')
            for r in result['anomaly_rows']
        ]

def main():
    return LightcurveApp("Lightcurve Explorer", "com.anaconda.demos.lightcurve")
```

## Building for every platform

One `pyproject.toml`, one source tree, six targets. Each platform section annotates what the output is and what tools are required.

In [ ]:
# Platform build matrix — see BUILDING.md for full per-platform walkthrough
print("""Platform     Command                                Output
──────────── ────────────────────────────────────── ──────────────────────────────────────
Development  briefcase dev                          Runs immediately, no build step

macOS        briefcase create macOS                 Scaffold + download support (~200 MB)
             briefcase build macOS                  .app bundle
             briefcase package macOS --adhoc-sign   .dmg installer
             → Requires: Xcode CLI tools
             → App Store: paid Apple Developer account + Xcode Organizer

Windows      briefcase create windows               Scaffold
             briefcase build windows                .exe + assets
             briefcase package windows --adhoc-sign .msi installer (WiX, auto-installed)
             → Requires: nothing extra — WiX bundled in Briefcase

Linux        briefcase create linux                 Scaffold
             briefcase build linux                  AppImage
             briefcase package linux --adhoc-sign   AppImage (runnable on any glibc distro)
             briefcase package linux -p deb         .deb (Ubuntu/Debian)
             briefcase package linux -p rpm         .rpm (Fedora/RHEL)
             → Requires: libgirepository, libcairo, libpango (apt auto-installs)

iOS          briefcase create iOS                   Xcode project
             briefcase run iOS                      iOS Simulator (free)
             briefcase run iOS -d 'iPhone 15 Pro'   Specific simulator
             briefcase package iOS                  → Xcode Organizer → App Store
             → Requires: macOS + Xcode. iOS numpy wheels: Anaconda adding in 2025.

Android      briefcase create android               Gradle project
             briefcase run android                  Android emulator
             briefcase package android              .aab (Play Store)
             briefcase package android -p apk       .apk (direct install)
             → Requires: Android Studio + JDK 17 (briefcase upgrade java)
             → Play Store: $25 one-time Google Play Developer account""")

Platform     Command                                Output
──────────── ────────────────────────────────────── ──────────────────────────────────────
Development  briefcase dev                          Runs immediately, no build step

macOS        briefcase create macOS                 Scaffold + download support (~200 MB)
             briefcase build macOS                  .app bundle
             briefcase package macOS --adhoc-sign   .dmg installer
             → Requires: Xcode CLI tools
             → App Store: paid Apple Developer account + Xcode Organizer

Windows      briefcase create windows               Scaffold
             briefcase build windows                .exe + assets
             briefcase package windows --adhoc-sign .msi installer (WiX, auto-installed)
             → Requires: nothing extra — WiX bundled in Briefcase

Linux        briefcase create linux                 Scaffold
             briefcase build linux                  AppImage
             briefcase p

## The Briefcase command lifecycle

```
briefcase dev       → run in development mode (no build, no scaffold, fastest iteration)
briefcase create    → scaffold the platform-specific project directory
briefcase build     → compile the app for the target
briefcase run       → run the built app
briefcase update    → push source code changes into an existing scaffold
briefcase package   → produce a distributable installer / archive
briefcase upgrade   → update Briefcase-managed tools (Java JDK, etc.)
```

Every command except `dev` accepts a platform argument: `macOS`, `windows`, `linux`, `iOS`, `android`. Without one, Briefcase targets the current OS.

**After changing app code:**
```bash
briefcase run --update        # push changes + run in one step
briefcase update -r           # update dependencies from pyproject.toml, then run
```

## Mobile: numpy + scikit-learn availability

The Anaconda OSS engineering team is actively building iOS and Android binary wheels for numpy and scikit-learn (tracked in BeeWare's monthly status updates, March 2025+).

The `app.py` in this demo handles this gracefully:

```python
try:
    import numpy as _np
    USE_NUMPY = True
except ImportError:
    USE_NUMPY = False   # mobile fallback: pure-Python synthesiser + 2.5σ threshold
```

When numpy isn't available, the light curve synthesiser falls back to Python's `random` + `math`, and anomaly detection uses a 2.5σ threshold instead of IsolationForest. The app runs on iOS and Android today — the detection method shown in the validation report tells you which path was taken.

Check current wheel availability:
```bash
pip index versions --index-url https://releases.beeware.org/mobile numpy
```

---
## Comparison

| | Option A — PyScript | Option B — BeeWare |
|---|---|---|
| **Delivery** | Static HTML file | Native OS app |
| **Runtime** | Browser (Pyodide / WASM) | Native CPython |
| **numpy / scikit-learn** | ✓ Pyodide bundle | ✓ Desktop; mobile: in progress |
| **matplotlib charts** | ✓ renders as PNG in DOM | Not used — Toga Table instead |
| **Data input** | Synthesised in-browser | Synthesised in-app |
| **Distribution** | Host anywhere (GitHub Pages, S3, CDN) | .dmg / .msi / AppImage / App Store |
| **Mobile** | MicroPython for lightweight | iOS + Android via Briefcase |
| **Offline after load** | ✓ | ✓ |
| **Best for** | Demos, education, web dashboards | Installable tools, offline desktop/mobile |
| **Anaconda connection** | Launched at PyCon US 2022 | Funded by Anaconda |
| **Time to first run** | ~5 s (Pyodide load, cached after) | `briefcase dev` ~3 s |
| **Lines of app code** | 688 (HTML + Python) | 529 (Python) + 106 (pyproject.toml) |

---

The pipeline is portable because its outputs are structured. The same `ValidationReport` fields — nulls, flux_std, phase_range, anomaly count, transit depth — appear in both the PyScript DOM and the Toga Label widgets. That structure came from `ingestion.py` in Module 01 and hasn't changed.